<a href="https://colab.research.google.com/github/Poornitha2023/24B11CS500_DriveReady_OPC_2026/blob/main/agenticAi_Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# Run this once to create the sheet. For later sessions, open the existing sheet.
sh = gc.create("Campus Stationery Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "qty", "pickup_date", "double_sided", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/113X0y9RuZ_WaviPq89KvHB_ytil3NSyswUqOg-CxSs0


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

# Sample catalogue and prices for the stationery shop
MENU = {
    "notebook": 60,
    "black-and-white print": 2,
    "spiral binding": 40
}

HOURS = {
    "monday": "8 am to 7 pm",
    "tuesday": "8 am to 7 pm",
    "wednesday": "8 am to 7 pm",
    "thursday": "8 am to 7 pm",
    "friday": "8 am to 7 pm",
    "saturday": "9 am to 5 pm",
    "sunday": "closed"
}

class Order(BaseModel):
    """A stationery or printing order collected from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["notebook", "black-and-white print", "spiral binding"] = Field(
        description="Product or service requested"
    )
    qty: int = Field(description="Number of notebooks, printed pages, or binding jobs", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    double_sided: bool = Field(description="True if the customer requested double-sided printing")

In [3]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Calculate the total price for a stationery product or printing service.

    Args:
        item: "notebook", "black-and-white print", or "spiral binding"
        qty: Number of items, printed pages, or binding jobs
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not in our catalogue. We offer: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the shop's opening hours for a day of the week, such as "saturday"."""
    hours = HOURS.get(day.lower().strip())
    return f"We are open {hours} on {day}." if hours else f"'{day}' is not a valid day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, double_sided: bool) -> str:
    """Save a confirmed stationery or printing order to the Orders sheet.
    Use only after collecting the customer's name, item, quantity, and pickup date."""
    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            double_sided=double_sided
        )
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct the details."

    total = MENU[order.item] * order.qty
    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.double_sided else "no",
        total
    ])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}. Total: Rs {total}."

In [4]:
from langchain.agents import create_agent

SYSTEM = """ROLE: You are a helpful assistant for Campus Stationery & Print Hub.
FACTS: The shop sells notebooks, black-and-white printing, and spiral binding.
       Double-sided printing is available at the same per-page price.
RULES: Always use get_price to calculate prices and is_open to check shop timings.
       Never guess prices or opening hours.
       Before log_order, collect the customer's name, item, quantity, and pickup date.
       For printing orders, also ask whether printing should be double-sided.
       If any required detail is missing, ask the customer instead of saving the order.
       After saving, repeat the order details and total.
FORMAT: Keep replies under 60 words. Use friendly English or Telugu to match the customer."""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

history = []

def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({
            "messages": history + [{"role": "user", "content": text}]
        })
        history = result["messages"]  # retain conversation history as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, something went wrong. Please try again."

# Example 1: Ask about opening hours
print(chat("Is the shop open on Sunday?"))

# Example 2: Ask for the price of multiple items
print(chat("How much will 4 notebooks and 25 black-and-white print pages cost?"))

# Example 3: Place a complete order
print(chat("I'm Ravi. I need 3 notebooks for pickup on 2 December."))

# Example 4: Place a printing order with an extra requirement
print(chat("My name is Anu. Please print 80 pages double-sided for pickup on 3 December."))

We’re closed on Sundays. If you need anything else, let us know!
4 notebooks cost Rs 240, 25 black‑and‑white print pages cost Rs 50.  
**Total: Rs 290**.
✅ Order saved!  
3 notebooks for Ravi, pickup 2 Dec. Total: Rs 180. Let me know if you need anything else!
✅ Order saved!  
80 double‑sided black‑and‑white pages for Anu, pickup 3 Dec. Total: Rs 160. Let me know if you need anything else!
